<a href="https://colab.research.google.com/github/bmulaw/DeepLearningNotebooks/blob/main/DeepLearning_TD3_Colab_Iris_VAE_BetaVAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab — Human Irises: Autoencoder, VAE and β-VAE
**Google Colab · Master's level (M1/M2) · Estimated duration: 2.5–3 hours**

**Learning objectives:** (1) explore a dataset of real iris images, (2) train a convolutional VAE, (3) visualize reconstructions and sample images, (4) compare β=1 and β=4, (5) inspect latent geometry using PCA and UMAP, (6) discuss the validity of statistical interpretations.

**Scope of use:** iris images are biometric data. This notebook uses a public sample for educational purposes, without personal identification or individual sex or gender prediction. Do not import personal images without permission.

Dataset: [Cornea Iris Dataset — Sarah Barrington (UC Berkeley)](https://github.com/sbarrington/cornea-iris-dataset), iris crops from real FFHQ portraits. Data source and usage rights: consult the repository before reuse.


## 0. Set up Colab
Enable **Runtime → Change runtime type → T4 GPU**, if available. The cells also run on a CPU with a small sample.


In [ ]:
!pip -q install umap-learn
import io, os, random, zipfile, requests, pathlib, math
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :',device)


Device : cuda


## 1. Load a real dataset of iris crops
Download the public repository. Keep only crops extracted from **real portraits**, excluding synthetic images. The repository may change: if the download fails, download the archive from GitHub, place it in `/content/`, and adjust `ROOT`.


In [ ]:
URL='https://github.com/sbarrington/cornea-iris-dataset/archive/refs/heads/main.zip'
ROOT=pathlib.Path('/content/iris_dataset')
ROOT.mkdir(parents=True,exist_ok=True)
try:
    r=requests.get(URL,timeout=120); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z: z.extractall(ROOT)
    print('Download successful:',round(len(r.content)/1e6,1),'MB')
except Exception as exc:
    print('Download failed:',exc)
    print('Alternative: place the repository ZIP in /content, then extract it into',ROOT)

all_imgs=[p for p in ROOT.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg','.bmp')]
# Real FFHQ iris images, excluding deepfake directories.
real_imgs=[p for p in all_imgs if 'irises_real_ffhq' in str(p).lower() and '512x512' not in str(p).lower()]
# The repository hierarchy may differ between versions.
if len(real_imgs)<30:
    real_imgs=[p for p in all_imgs if '512x512_irises_real_ffhq' in str(p).lower()]
print('Real images found:',len(real_imgs))
if len(real_imgs)<30:
    raise RuntimeError('Fewer than 30 images. Check the repository structure and licenses / change ROOT.')
random.shuffle(real_imgs)
MAX_IMAGES=2400
paths=real_imgs[:MAX_IMAGES]
print('Selected images:',len(paths))


In [ ]:
fig,ax=plt.subplots(2,6,figsize=(12,4))
for a,p in zip(ax.flat, paths[:12]):
    a.imshow(Image.open(p).convert('RGB')); a.axis('off')
plt.suptitle('Iris crops (real images)'); plt.tight_layout(); plt.show()


**Discussion (5 min):** do these crops contain only the iris? Which details might provide statistical shortcuts (reflections, eyelashes, lighting, framing)?


## 2. Prepare the data
RGB images at 64×64; pixel values normalized to [0,1]. To guard against memorization, maintain a training/validation split. **Important:** when participant identifiers are available, split by person rather than by image; the selected repository is not guaranteed to provide these identifiers.


In [ ]:
IMG_SIZE=64
transform=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor()])
class IrisDataset(Dataset):
    def __init__(self, paths): self.paths=paths
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im:
            x=transform(im.convert('RGB'))
        return x

ds=IrisDataset(paths)
n_train=int(.85*len(ds)); n_val=len(ds)-n_train
train_ds,val_ds=random_split(ds,[n_train,n_val],generator=torch.Generator().manual_seed(SEED))
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,num_workers=2,pin_memory=device.type=='cuda')
val_loader=DataLoader(val_ds,batch_size=64,shuffle=False,num_workers=2,pin_memory=device.type=='cuda')
print('train / val :',n_train,n_val)


## 3. Build a convolutional VAE

\(q_\phi(z\mid x)=\mathcal N(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x)))\)

**Reparameterization trick:** \(z=\mu+\exp(0.5\log\sigma^2)\odot\varepsilon\), \(\varepsilon\sim\mathcal N(0,I)\).

\(\mathcal L_\beta=\mathbb E[\|x-\hat x\|_2^2]+\beta D_{KL}(q_\phi(z\mid x)\|\mathcal N(0,I))\).

We use the **sum per image** of reconstruction errors, then average over the batch, to compare the reconstruction and KL contributions. The effect of a given β depends on this normalization convention.


In [ ]:
LATENT_DIM=12
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=12):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Conv2d(3,32,4,2,1),nn.ReLU(),
            nn.Conv2d(32,64,4,2,1),nn.ReLU(),
            nn.Conv2d(64,128,4,2,1),nn.ReLU(),
            nn.Conv2d(128,128,4,2,1),nn.ReLU())
        self.fc_mu=nn.Linear(128*4*4,latent_dim)
        self.fc_logvar=nn.Linear(128*4*4,latent_dim)
        self.decoder_fc=nn.Linear(latent_dim,128*4*4)
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(128,128,4,2,1),nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1),nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1),nn.Sigmoid())
    def encode(self,x):
        h=self.encoder(x).flatten(1)
        return self.fc_mu(h),self.fc_logvar(h)
    def reparameterize(self,mu,logvar):
        eps=torch.randn_like(mu)
        return mu+torch.exp(.5*logvar)*eps
    def decode(self,z):
        return self.decoder(self.decoder_fc(z).reshape(-1,128,4,4))
    def forward(self,x):
        mu,logvar=self.encode(x)
        return self.decode(self.reparameterize(mu,logvar)),mu,logvar

def vae_loss(x_hat,x,mu,logvar,beta):
    rec=((x_hat-x)**2).flatten(1).sum(1).mean()
    kl=(-0.5*(1+logvar-mu.pow(2)-logvar.exp()).sum(1)).mean()
    return rec+beta*kl,rec.detach(),kl.detach()
model=ConvVAE(LATENT_DIM).to(device)
print('Parameters:',sum(p.numel() for p in model.parameters()))


## 4. Train β-VAEs with β=1 and β=4
Use the same model architecture, training dataset, and epoch budget. Results depend on training time and the dataset.


In [ ]:
EPOCHS=12  # increase to 25 to improve reconstructions
BETAS=(1.,4.)
def fit(beta,epochs=EPOCHS):
    model=ConvVAE(LATENT_DIM).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    history={'train':[],'val':[],'rec':[],'kl':[]}
    for epoch in range(epochs):
        model.train(); train_total=0; count=0
        for x in train_loader:
            x=x.to(device); opt.zero_grad()
            xh,mu,lv=model(x)
            loss,_,_=vae_loss(xh,x,mu,lv,beta)
            loss.backward();opt.step()
            train_total+=loss.item()*len(x);count+=len(x)
        model.eval(); vl=vr=vk=0; vn=0
        with torch.no_grad():
            for x in val_loader:
                x=x.to(device); xh,mu,lv=model(x)
                loss,rec,kl=vae_loss(xh,x,mu,lv,beta)
                vl+=loss.item()*len(x);vr+=rec.item()*len(x);vk+=kl.item()*len(x);vn+=len(x)
        for k,v in [('train',train_total/count),('val',vl/vn),('rec',vr/vn),('kl',vk/vn)]:history[k].append(v)
        print(f'beta={beta:g} epoch {epoch+1:02d}/{epochs} | val={vl/vn:.1f} | rec={vr/vn:.1f} | KL={vk/vn:.1f}')
    return model,history
models={}; histories={}
for beta in BETAS:
    models[beta],histories[beta]=fit(beta)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for beta,h in histories.items():
    for ax,key in zip(axes,['val','rec','kl']): ax.plot(h[key],label=f'β={beta:g}')
for ax,title in zip(axes,['Total loss (val)','Reconstruction (val)','KL (val)']):
    ax.set_title(title);ax.set_xlabel('Epoch');ax.legend();ax.grid(alpha=.2)
plt.tight_layout();plt.show()


**Question:** does β=4 consistently improve generation? What quantitative trade-off do the curves reveal?


## 5. Inspect reconstructions and generate new irises
The generated outputs are **synthetic images**, not a person's authentic iris. They must not be used as biometric reference samples.


In [ ]:
from torchvision.utils import make_grid
from torchvision.transforms.functional import to_pil_image
xb=next(iter(val_loader))[:8].to(device)
fig,axs=plt.subplots(3,8,figsize=(14,6))
for i in range(8): axs[0,i].imshow(to_pil_image(xb[i].cpu()));axs[0,i].axis('off')
for row,beta in [(1,1.),(2,4.)]:
    models[beta].eval()
    with torch.no_grad():mu,_=models[beta].encode(xb); xrec=models[beta].decode(mu).cpu()
    for i in range(8):axs[row,i].imshow(to_pil_image(xrec[i]));axs[row,i].axis('off')
for ax,label in zip(axs[:,0],['Original','VAE β=1','β-VAE β=4']):ax.set_ylabel(label,rotation=90)
plt.suptitle('Reconstructions from µ (without noise)');plt.tight_layout();plt.show()


In [ ]:
fig,axs=plt.subplots(2,8,figsize=(14,4))
for row,beta in enumerate(BETAS):
    models[beta].eval()
    with torch.no_grad():
        z=torch.randn(8,LATENT_DIM,device=device)
        generated=models[beta].decode(z).cpu()
    for i in range(8):axs[row,i].imshow(to_pil_image(generated[i]));axs[row,i].axis('off')
    axs[row,0].set_ylabel(f'β={beta:g}')
plt.suptitle('Samples z ~ N(0,I)');plt.tight_layout();plt.show()


## 6. Explore latent space: PCA and UMAP
For a stable analysis, plot the **means µ(x)** rather than random samples of \(z\). Color points by **pupil size approximated using a central-darkness indicator**, a simple visual proxy with no clinical significance.


In [ ]:
import umap
@torch.no_grad()
def extract_latents(model,loader):
    model.eval();Z=[]; scores=[]
    for x in loader:
        mu,_=model.encode(x.to(device));Z.append(mu.cpu().numpy())
        # Central luminance proxy, NOT a medical measurement of pupil diameter
        h=x.shape[-1]; patch=x[:,:,h//2-5:h//2+5,h//2-5:h//2+5]
        scores.extend(patch.mean((1,2,3)).numpy().tolist())
    return np.concatenate(Z),np.array(scores)
fig,axs=plt.subplots(2,2,figsize=(12,10))
for row,beta in enumerate(BETAS):
    Z,brightness=extract_latents(models[beta],val_loader)
    zp=PCA(n_components=2).fit_transform(Z)
    zu=umap.UMAP(n_neighbors=min(15,len(Z)-1),min_dist=.2,random_state=SEED).fit_transform(Z)
    for col,(emb,method) in enumerate([(zp,'PCA'),(zu,'UMAP')]):
        sc=axs[row,col].scatter(emb[:,0],emb[:,1],c=brightness,s=13,cmap='viridis',alpha=.8)
        axs[row,col].set_title(f'{method} — β={beta:g}');axs[row,col].set_xlabel('Dimension 1');axs[row,col].set_ylabel('Dimension 2')
        fig.colorbar(sc,ax=axs[row,col],label='Central luminance (proxy)')
plt.tight_layout();plt.show()


## 7. Semantic trajectories — modify one latent dimension
Fix all dimensions at \(µ(x)\), except one that is varied over an interval. A visible change **does not prove** that this coordinate is a causal biological variable.


In [ ]:
beta=1.;m=models[beta];m.eval()
example=next(iter(val_loader))[0:1].to(device)
with torch.no_grad():mu,_=m.encode(example)
DIM_TO_SWEEP=0
steps=np.linspace(-2.5,2.5,9)
fig,ax=plt.subplots(1,len(steps),figsize=(15,2.4))
with torch.no_grad():
    z=mu.repeat(len(steps),1)
    z[:,DIM_TO_SWEEP]=torch.as_tensor(steps,dtype=z.dtype,device=z.device)
    im=m.decode(z).cpu()
for i,v in enumerate(steps):ax[i].imshow(to_pil_image(im[i]));ax[i].set_title(f'{v:.1f}');ax[i].axis('off')
plt.suptitle(f'Latent traversal — dim {DIM_TO_SWEEP}, β={beta:g}');plt.tight_layout();plt.show()


## 8. Interpolate between two irises (latent space navigation)


In [ ]:
x=next(iter(val_loader))[:2].to(device)
model=models[1.];model.eval()
with torch.no_grad():
    mus,_=model.encode(x)
    t=torch.linspace(0,1,9,device=device).unsqueeze(1)
    z=(1-t)*mus[0:1]+t*mus[1:2]
    interpolated=model.decode(z).cpu()
fig,ax=plt.subplots(1,9,figsize=(15,2.5))
for j in range(9):ax[j].imshow(to_pil_image(interpolated[j]));ax[j].set_title(f't={j/8:.2f}');ax[j].axis('off')
plt.suptitle('Linear interpolation in latent space');plt.tight_layout();plt.show()


## 9. Mini research project: hypotheses and controls

**The current data have no validated male/female labels, so no group association can be measured here.** To study this topic at an aggregate level, the [ND-GFI — Notre Dame](https://cvrl.nd.edu/projects/data/) research dataset provides specific metadata and requires a license / access authorization. A rigorous study would require participant-level splits and acquisition controls, and would not justify concluding that an iris reveals an individual's gender.

Here are four exercises you can carry out directly in this notebook:

1. Vary β across {0.2, 1, 4, 10}; compare reconstruction and KL.
2. Change `LATENT_DIM` across {2, 8, 12, 24}; measure how reconstructions change.
3. Test UMAP stability with several random seeds; compare with PCA.
4. Select two irises, interpolate between their encodings, and describe the visual attributes that change (contrast, apparent center size, texture, reflections).

**Critical discussion:** which artifacts (framing, skin, reflections, lighting, capture hardware) could be mistaken for a biological factor? How could you design a validation procedure that makes the hypothesis falsifiable?


## 10. Review questions / report

- Why does the VAE have two outputs, µ and log σ²?
- Why does the KL term penalize a latent distribution that is too far from the N(0,I) prior?
- Does a larger β necessarily imply better factorization?
- Why is visual separation in UMAP not statistical evidence?
- Why is independent validation by person and, ideally, by sensor necessary?
- State a measurable hypothesis, a negative control, and a failure criterion.

**Student deliverable:** plots of learning curves, reconstructions, generated samples, PCA/UMAP projections, a latent traversal, and a critical interpretation (1 page).
